# Part 8 — Towards molecules: graph-transformer energy model & alanine dipeptide
Paper §3.2 (Eq. 15), §5, App. A.5, B.3. See [README.md](README.md).

Sections 1–3 run anywhere. Section 4 needs the alanine-dipeptide trajectory (see README) and, realistically, a GPU
for the paper's training budget.

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent            # notebooks live one level below the project root
sys.path.insert(0, str(ROOT))
import numpy as np, torch, matplotlib.pyplot as plt
torch.set_default_dtype(torch.float64)      # float64: the FP estimator uses finite differences with sigma=1e-4
torch.set_num_threads(8)
from scoremd import sde, models, losses, sampling, simulate, metrics, potentials, plotting, experiments as ex
%matplotlib inline
from scoremd import molecules
from scoremd.train import train

## 1. The architecture: invariances and conservativeness

In [2]:
net = molecules.GraphEnergyNet(molecules.ALA2_CG_ATOM_TYPES, hidden=96, layers=2)
print('parameters:', sum(p.numel() for p in net.parameters()))
z = torch.randn(8, 15); t = torch.full((8,), 0.1)
shift = torch.tensor([3., -1., 2.]).repeat(5)
print('translation invariance  |Δ log p| =', float((net.log_p(z, t) - net.log_p(z + shift, t)).abs().max()))
s = net.score(z, t)
print('net force (sum over atoms) =', float(s.view(8, 5, 3).sum(1).abs().max()), '(0 because of translation invariance)')
print('rotation: |Δ log p| =', float((net.log_p(z, t) - net.log_p(molecules.random_rotation(z), t)).abs().max()),
      '-> NOT invariant by construction; learned from augmented data')

parameters: 206211
translation invariance  |Δ log p| = 2.220446049250313e-16
net force (sum over atoms) = 5.204170427930421e-18 (0 because of translation invariance)
rotation: |Δ log p| = 0.027820425549243633 -> NOT invariant by construction; learned from augmented data


## 2. Rotation equivariance is *learned* via augmentation
Quick demo on a synthetic rigid-ish 5-bead chain (not real data): train briefly with and without random rotations
and measure how much `log p` changes under rotation.

In [3]:
torch.manual_seed(0)
base = torch.tensor([[0, 0, 0], [1.3, 0, 0], [1.9, 1.2, 0], [3.3, 1.3, 0.4], [3.9, 2.4, 0.9]])
fake = (base[None] + 0.08 * torch.randn(4000, 5, 3)); fake = fake - fake.mean(1, keepdim=True)
fake = (fake / fake.std()).reshape(4000, 15)
def rot_gap(m):
    zz = molecules.random_rotation(fake[:256]); tt = torch.full((256,), 1e-3)
    return float((m.log_p(zz, tt) - m.log_p(molecules.random_rotation(zz), tt)).abs().mean())
for aug in [None, molecules.random_rotation]:
    m = molecules.GraphEnergyNet(molecules.ALA2_CG_ATOM_TYPES, hidden=32, layers=2)
    train(m, fake, steps=300, lr=1e-3, augment=aug, verbose=False); m.eval()
    print(f'augmentation={aug is not None}:  mean |log p(Rz) - log p(z)| = {rot_gap(m):.3f}')

augmentation=False:  mean |log p(Rz) - log p(z)| = 1.008
augmentation=True:  mean |log p(Rz) - log p(z)| = 0.038


## 3. CG mapping and dihedrals
Alanine dipeptide → 5 beads `[C, N, CA, C, N]` (App. B.3); φ = C–N–CA–C, ψ = N–CA–C–N are exactly the two dihedrals
of this chain, so the Ramachandran plot can be computed from the CG beads.

In [4]:
print('CG indices into the 22-atom topology:', molecules.ALA2_CG_INDICES, '(verify against your topology!)')
demo = torch.randn(3, 5, 3)
print('phi/psi (rad) of random beads:', molecules.ala2_phi_psi_cg(demo))

CG indices into the 22-atom topology: [4, 6, 8, 14, 16] (verify against your topology!)
phi/psi (rad) of random beads: tensor([[ 1.3939,  2.3165],
        [-0.4848,  1.6629],
        [ 1.7009,  0.8678]])


## 4. Alanine-dipeptide pipeline (needs data)
Put an array of all-atom coordinates with shape `(n_frames, 22, 3)` (in nm or Å) at `DATA_PATH`, e.g. exported from
the bgmol implicit-solvent dataset or the ScoreMD repository. The cell is skipped if the file does not exist.

Paper settings (Table 3, "Both"): expert 0 on `(0, 0.1)`, hidden 96, 2 layers, α = 1e-3, 7000 epochs on 50k frames,
batch 1024, simulation at `t = 1e-5`. The defaults below are a *tiny* CPU version — scale up on a GPU.

In [5]:
import pathlib
DATA_PATH = ROOT / 'data' / 'ala2_coords.npy'
TRAIN_STEPS, ALPHA, SIM_STEPS = 2000, 1e-3, 5000

if not DATA_PATH.exists():
    print(f'No data at {DATA_PATH} - skipping. See README.md section 5 for how to get it.')
else:
    allatom = torch.tensor(np.load(DATA_PATH))
    cg = allatom[:, molecules.ALA2_CG_INDICES]                  # (n, 5, 3)
    cg = cg - cg.mean(1, keepdim=True)                           # remove centre of mass
    scale = cg.std(); z = (cg / scale).reshape(len(cg), 15)      # isotropic normalisation
    ref_angles = molecules.ala2_phi_psi_cg(z)

    expert0 = molecules.GraphEnergyNet(molecules.ALA2_CG_ATOM_TYPES, hidden=96, layers=2)
    train(expert0, z, steps=TRAIN_STEPS, batch_size=256, lr=3e-4, t_range=(1e-5, 0.1), fp_alpha=ALPHA,
          augment=molecules.random_rotation, log_every=200)
    torch.save(expert0.state_dict(), ex.RESULTS / 'ala2_expert0.pt')

    # Langevin simulation in normalised units with the learned force (expert 0 at t = 1e-5)
    idx = torch.randint(0, len(z), (50,))
    traj = simulate.langevin(simulate.model_force(expert0), z[idx], n_steps=SIM_STEPS, dt=0.01, gamma=1.0, save_every=10)
    sim = traj[len(traj) // 10:].reshape(-1, 15)
    sim_angles = molecules.ala2_phi_psi_cg(sim)

    R = [[-np.pi, np.pi], [-np.pi, np.pi]]
    fig, ax = plt.subplots(1, 2, figsize=(9, 4))
    plotting.free_energy_hist(ax[0], ref_angles, R, bins=64, title='reference (phi, psi)')
    plotting.free_energy_hist(ax[1], sim_angles, R, bins=64, title='expert 0 simulation')
    print('sim PMF error (64 bins):', metrics.pmf_error(sim_angles, ref_angles, 64, R))
    bl_ref = molecules.bond_lengths_cg(cg); bl_sim = molecules.bond_lengths_cg(sim.view(-1, 5, 3) * scale)
    print('bond length mean/std  ref:', bl_ref.mean(0), bl_ref.std(0), '\n                      sim:', bl_sim.mean(0), bl_sim.std(0))

No data at /Users/gamzekantar/Desktop/PML/data/ala2_coords.npy - skipping. See README.md section 5 for how to get it.


## Next steps for a full reproduction
1. Add the two large-`t` experts (`ScoreNet`-style graph transformers) to get iid sampling → Table 11's iid column.
2. Compare Diffusion (α = 0), Mixture, Fokker–Planck and Both; record sim PMF next to Table 11.
3. Proteins (Chignolin/BBA) require the Lindorff-Larsen et al. (2011) trajectories (D. E. Shaw Research — access on
   request), TICA (`deeptime`) and multi-GPU training — follow the official ScoreMD repo for those.